In [2]:
import pandas as pd

csv_path = 'C:\\Users\\athar\\smartgrowth-ai\\data\\WA_Fn-UseC_-Telco-Customer-Churn.csv'
df = pd.read_csv(csv_path)

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
# Data Overview and Basic Info
print("🔍 SmartGrowth AI - Churn Prediction Model Development")
print("="*60)

print(f"\n📊 Dataset Overview:")
print(f"  - Shape: {df.shape}")
print(f"  - Columns: {list(df.columns)}")
print(f"  - Missing values: {df.isnull().sum().sum()}")

# Display basic statistics
print(f"\n📈 Target Variable (Churn):")
churn_counts = df['Churn'].value_counts()
print(f"  - No Churn: {churn_counts['No']} ({churn_counts['No']/len(df):.1%})")
print(f"  - Churn: {churn_counts['Yes']} ({churn_counts['Yes']/len(df):.1%})")

# Check data types
print(f"\n🔍 Data Types:")
print(df.dtypes)

df.head()

🔍 SmartGrowth AI - Churn Prediction Model Development

📊 Dataset Overview:
  - Shape: (7043, 21)
  - Columns: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']
  - Missing values: 0

📈 Target Variable (Churn):
  - No Churn: 5174 (73.5%)
  - Churn: 1869 (26.5%)

🔍 Data Types:
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
Pa

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [7]:
pip install scikit-learn

  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.1 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.1 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.1 MB 648.0 kB/s eta 0:00:12
   ----- ---------------------------------- 1.0/8.1 MB 1.0 MB/s eta 0:00:07
   --------- ------------------------------ 1.8/8.1 MB 1.6 MB/s eta 0:00:04
   ------------ --------------------------- 2.6/8.1 MB 2.0 MB/s eta 0:00:03
   ---------------- ----------------------- 3.4/8.1 MB 2.3 MB/s eta 0:00:03
   -------------------- ------------------- 4.2/8.1 MB 2.4 MB/s eta 0:00:02
   ------------------------ --------------- 5.0/8.1 MB 2.6 MB/s eta 0:00:02
   ---------------------------- ----------- 5.8/8.1 


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
# Data Cleaning and Preprocessing
import numpy as np
from sklearn.preprocessing import LabelEncoder

print("🛠️ Data Cleaning and Preprocessing")
print("="*40)

# Create a clean copy 
df_clean = df.copy()

# Handle TotalCharges - some values are spaces, need to convert to numeric
print("📋 Cleaning TotalCharges column...")
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'], errors='coerce')

# Check missing values in TotalCharges
missing_total = df_clean['TotalCharges'].isna().sum()
print(f"  - Missing TotalCharges: {missing_total}")

# Fill missing values with monthly charges * tenure (reasonable assumption)
if missing_total > 0:
    df_clean['TotalCharges'] = df_clean['TotalCharges'].fillna(
        df_clean['MonthlyCharges'] * df_clean['tenure']
    )
    print(f"  - Filled missing values with MonthlyCharges * tenure")

# Convert categorical variables to binary
print("\n🔄 Converting categorical variables...")
binary_maps = {
    'gender': {'Female': 0, 'Male': 1},
    'Partner': {'No': 0, 'Yes': 1},
    'Dependents': {'No': 0, 'Yes': 1},
    'Churn': {'No': 0, 'Yes': 1}
}

for column, mapping in binary_maps.items():
    df_clean[column] = df_clean[column].map(mapping)
    print(f"  - {column}: {list(mapping.keys())} → {list(mapping.values())}")

# Encode Contract type (ordinal: Month-to-month < One year < Two year)
contract_map = {'Month-to-month': 0, 'One year': 1, 'Two year': 2}
df_clean['Contract'] = df_clean['Contract'].map(contract_map)
print(f"  - Contract: Month-to-month=0, One year=1, Two year=2")

print(f"\n✅ Data cleaning complete!")
print(f"  - Final shape: {df_clean.shape}")
print(f"  - Missing values: {df_clean.isnull().sum().sum()}")

df_clean.head()

🛠️ Data Cleaning and Preprocessing
📋 Cleaning TotalCharges column...
  - Missing TotalCharges: 11
  - Filled missing values with MonthlyCharges * tenure

🔄 Converting categorical variables...
  - gender: ['Female', 'Male'] → [0, 1]
  - Partner: ['No', 'Yes'] → [0, 1]
  - Dependents: ['No', 'Yes'] → [0, 1]
  - Churn: ['No', 'Yes'] → [0, 1]
  - Contract: Month-to-month=0, One year=1, Two year=2

✅ Data cleaning complete!
  - Final shape: (7043, 21)
  - Missing values: 0


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,0,0,1,0,1,No,No phone service,DSL,No,...,No,No,No,No,0,Yes,Electronic check,29.85,29.85,0
1,5575-GNVDE,1,0,0,0,34,Yes,No,DSL,Yes,...,Yes,No,No,No,1,No,Mailed check,56.95,1889.50,0
2,3668-QPYBK,1,0,0,0,2,Yes,No,DSL,Yes,...,No,No,No,No,0,Yes,Mailed check,53.85,108.15,1
3,7795-CFOCW,1,0,0,0,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,1,No,Bank transfer (automatic),42.30,1840.75,0
4,9237-HQITU,0,0,0,0,2,Yes,No,Fiber optic,No,...,No,No,No,No,0,Yes,Electronic check,70.70,151.65,1


In [18]:
%pip install matplotlib seaborn


  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ------- -------------------------------- 1.6/8.3 MB 8.5 MB/s eta 0:00:01
   ---------------- ----------------------- 3.4/8.3 MB 8.4 MB/s eta 0:00:01
   ------------------------- -------------- 5.2/8.3 MB 8.2 MB/s eta 0:00:01
   -------------------------------- ------- 6.8/8.3 MB 8.2 MB/s eta 0:00:01
   ---------------------------------------- 8.3/8.3 MB 7.9 MB/s  0:00:01
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   --------------------------- ------------ 1.6/2.3 MB 7.6 MB/s eta 0:00:01
   ---------------------------------------- 2.3/2.3 MB 6.8 MB/s  0:00:00

   ---------------------------------------- 0/7 [pyparsing]
   ----- ------------------------------


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
# Exploratory Data Analysis & Feature Engineering
import matplotlib.pyplot as plt
import seaborn as sns

print("📊 Exploratory Data Analysis & Feature Engineering")
print("="*50)

# Basic correlation analysis
print("🔍 Correlation with Churn:")
numeric_cols = ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 
                'Contract', 'MonthlyCharges', 'TotalCharges']

correlations = df_clean[numeric_cols + ['Churn']].corr()['Churn'].sort_values(ascending=False)
print(correlations.drop('Churn'))

# Feature Engineering - Create meaningful features
print("\n⚙️ Creating engineered features...")

# 1. Average charges per month
df_clean['AvgChargesPerMonth'] = df_clean['TotalCharges'] / (df_clean['tenure'] + 1)  # +1 to avoid division by zero
print("  ✅ AvgChargesPerMonth = TotalCharges / tenure")

# 2. Customer segment based on tenure
def categorize_tenure(tenure):
    if tenure <= 12:
        return 0  # New customer
    elif tenure <= 36:
        return 1  # Regular customer  
    else:
        return 2  # Long-term customer

df_clean['TenureGroup'] = df_clean['tenure'].apply(categorize_tenure)
print("  ✅ TenureGroup: 0=New(≤12), 1=Regular(13-36), 2=Long-term(>36)")

# 3. High value customer indicator
monthly_median = df_clean['MonthlyCharges'].median()
df_clean['IsHighValue'] = (df_clean['MonthlyCharges'] > monthly_median).astype(int)
print(f"  ✅ IsHighValue: Monthly charges > ${monthly_median:.2f}")

# 4. Senior citizen with family
df_clean['SeniorWithFamily'] = (df_clean['SeniorCitizen'] & (df_clean['Partner'] | df_clean['Dependents'])).astype(int)
print("  ✅ SeniorWithFamily: Senior citizen + (Partner OR Dependents)")

# Display feature importance insights
print(f"\n📈 Key Insights:")
print(f"  - Churn rate by contract: Month-to-month={df_clean[df_clean['Contract']==0]['Churn'].mean():.1%}, "
      f"One year={df_clean[df_clean['Contract']==1]['Churn'].mean():.1%}, "
      f"Two year={df_clean[df_clean['Contract']==2]['Churn'].mean():.1%}")
print(f"  - Churn rate by tenure group: New={df_clean[df_clean['TenureGroup']==0]['Churn'].mean():.1%}, "
      f"Regular={df_clean[df_clean['TenureGroup']==1]['Churn'].mean():.1%}, "
      f"Long-term={df_clean[df_clean['TenureGroup']==2]['Churn'].mean():.1%}")

df_clean.head()

📊 Exploratory Data Analysis & Feature Engineering
🔍 Correlation with Churn:
MonthlyCharges    0.193356
SeniorCitizen     0.150889
gender           -0.008612
Partner          -0.150448
Dependents       -0.164221
TotalCharges     -0.198324
tenure           -0.352229
Contract         -0.396713
Name: Churn, dtype: float64

⚙️ Creating engineered features...
  ✅ AvgChargesPerMonth = TotalCharges / tenure
  ✅ TenureGroup: 0=New(≤12), 1=Regular(13-36), 2=Long-term(>36)
  ✅ IsHighValue: Monthly charges > $70.35
  ✅ SeniorWithFamily: Senior citizen + (Partner OR Dependents)

📈 Key Insights:
  - Churn rate by contract: Month-to-month=42.7%, One year=11.3%, Two year=2.8%
  - Churn rate by tenure group: New=47.4%, Regular=25.5%, Long-term=11.9%


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,AvgChargesPerMonth,TenureGroup,IsHighValue,SeniorWithFamily
0,7590-VHVEG,0,0,1,0,1,No,No phone service,DSL,No,...,0,Yes,Electronic check,29.85,29.85,0,14.925000,0,0,0
1,5575-GNVDE,1,0,0,0,34,Yes,No,DSL,Yes,...,1,No,Mailed check,56.95,1889.50,0,53.985714,1,0,0
2,3668-QPYBK,1,0,0,0,2,Yes,No,DSL,Yes,...,0,Yes,Mailed check,53.85,108.15,1,36.050000,0,0,0
3,7795-CFOCW,1,0,0,0,45,No,No phone service,DSL,Yes,...,1,No,Bank transfer (automatic),42.30,1840.75,0,40.016304,2,0,0
4,9237-HQITU,0,0,0,0,2,Yes,No,Fiber optic,No,...,0,Yes,Electronic check,70.70,151.65,1,50.550000,0,1,0


In [20]:
# Machine Learning Model Training
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("🤖 Machine Learning Model Training")
print("="*40)

# Prepare features and target
feature_columns = ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 
                   'Contract', 'MonthlyCharges', 'TotalCharges', 'AvgChargesPerMonth',
                   'TenureGroup', 'IsHighValue', 'SeniorWithFamily']

X = df_clean[feature_columns]
y = df_clean['Churn']

print(f"📊 Dataset for ML:")
print(f"  - Features: {len(feature_columns)}")
print(f"  - Samples: {len(X)}")
print(f"  - Positive class (Churn): {y.sum()} ({y.mean():.1%})")

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n📝 Train/Test Split:")
print(f"  - Training samples: {len(X_train)}")
print(f"  - Testing samples: {len(X_test)}")
print(f"  - Training churn rate: {y_train.mean():.1%}")
print(f"  - Testing churn rate: {y_test.mean():.1%}")

# Scale features for logistic regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\n🎯 Training Multiple Models...")

# Dictionary to store models and results
models = {}
results = {}

# 1. Logistic Regression
print("  📈 Training Logistic Regression...")
lr_model = LogisticRegression(random_state=42, class_weight='balanced', max_iter=1000)
lr_model.fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)
lr_pred_proba = lr_model.predict_proba(X_test_scaled)[:, 1]

models['Logistic Regression'] = lr_model
results['Logistic Regression'] = {
    'accuracy': accuracy_score(y_test, lr_pred),
    'precision': precision_score(y_test, lr_pred),
    'recall': recall_score(y_test, lr_pred), 
    'f1': f1_score(y_test, lr_pred),
    'auc': roc_auc_score(y_test, lr_pred_proba)
}

# 2. Random Forest
print("  🌳 Training Random Forest...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_pred_proba = rf_model.predict_proba(X_test)[:, 1]

models['Random Forest'] = rf_model
results['Random Forest'] = {
    'accuracy': accuracy_score(y_test, rf_pred),
    'precision': precision_score(y_test, rf_pred),
    'recall': recall_score(y_test, rf_pred),
    'f1': f1_score(y_test, rf_pred),
    'auc': roc_auc_score(y_test, rf_pred_proba)
}

# 3. Gradient Boosting
print("  🚀 Training Gradient Boosting...")
gb_model = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb_model.fit(X_train, y_train)
gb_pred = gb_model.predict(X_test)
gb_pred_proba = gb_model.predict_proba(X_test)[:, 1]

models['Gradient Boosting'] = gb_model
results['Gradient Boosting'] = {
    'accuracy': accuracy_score(y_test, gb_pred),
    'precision': precision_score(y_test, gb_pred),
    'recall': recall_score(y_test, gb_pred),
    'f1': f1_score(y_test, gb_pred),
    'auc': roc_auc_score(y_test, gb_pred_proba)
}

print(f"\n✅ Model training complete!")

# Display results comparison
print(f"\n📊 Model Performance Comparison:")
print("-" * 80)
print(f"{'Model':<20} {'Accuracy':<10} {'Precision':<11} {'Recall':<8} {'F1':<8} {'AUC':<8}")
print("-" * 80)

for model_name, metrics in results.items():
    print(f"{model_name:<20} {metrics['accuracy']:<10.3f} {metrics['precision']:<11.3f} "
          f"{metrics['recall']:<8.3f} {metrics['f1']:<8.3f} {metrics['auc']:<8.3f}")

# Find best model
best_model_name = max(results.keys(), key=lambda x: results[x]['auc'])
best_model = models[best_model_name]

print(f"\n🏆 Best Model: {best_model_name} (AUC: {results[best_model_name]['auc']:.3f})")

🤖 Machine Learning Model Training
📊 Dataset for ML:
  - Features: 12
  - Samples: 7043
  - Positive class (Churn): 1869 (26.5%)

📝 Train/Test Split:
  - Training samples: 5634
  - Testing samples: 1409
  - Training churn rate: 26.5%
  - Testing churn rate: 26.5%

🎯 Training Multiple Models...
  📈 Training Logistic Regression...
  🌳 Training Random Forest...
  🚀 Training Gradient Boosting...

✅ Model training complete!

📊 Model Performance Comparison:
--------------------------------------------------------------------------------
Model                Accuracy   Precision   Recall   F1       AUC     
--------------------------------------------------------------------------------
Logistic Regression  0.724      0.488       0.799    0.606    0.835   
Random Forest        0.767      0.580       0.439    0.499    0.796   
Gradient Boosting    0.796      0.656       0.489    0.560    0.836   

🏆 Best Model: Gradient Boosting (AUC: 0.836)


In [ ]:
# Model Interpretation & Feature Importance
import pandas as pd

print("🔍 Model Interpretation & Feature Importance")
print("="*50)

# Feature importance from best model
if 'Random Forest' in best_model_name or 'Gradient Boosting' in best_model_name:
    feature_importance = pd.DataFrame({
        'feature': feature_columns,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print(f"📈 Feature Importance ({best_model_name}):")
    print("-" * 40)
    for _, row in feature_importance.head(10).iterrows():
        print(f"  {row['feature']:<20}: {row['importance']:.3f}")

elif 'Logistic Regression' in best_model_name:
    # For logistic regression, use coefficient magnitude
    feature_importance = pd.DataFrame({
        'feature': feature_columns,
        'importance': abs(best_model.coef_[0])
    }).sort_values('importance', ascending=False)
    
    print(f"📈 Feature Importance ({best_model_name}):")
    print("-" * 40)
    for _, row in feature_importance.head(10).iterrows():
        print(f"  {row['feature']:<20}: {row['importance']:.3f}")

# Business insights from model
print(f"\n💡 Key Business Insights:")

# Churn probability by customer segments
if 'Random Forest' in best_model_name or 'Gradient Boosting' in best_model_name:
    # Use the original features for prediction
    test_probs = best_model.predict_proba(X_test)[:, 1]
else:
    # Use scaled features for logistic regression
    test_probs = best_model.predict_proba(X_test_scaled)[:, 1]

# Combine with test data for analysis
test_df = X_test.copy()
test_df['actual_churn'] = y_test
test_df['predicted_churn_prob'] = test_probs
test_df['risk_level'] = pd.cut(test_probs, 
                               bins=[0, 0.3, 0.7, 1.0], 
                               labels=['Low Risk', 'Medium Risk', 'High Risk'])

print("\n📊 Risk Segmentation Analysis:")
risk_analysis = test_df.groupby('risk_level').agg({
    'actual_churn': ['count', 'mean'],
    'predicted_churn_prob': 'mean',
    'MonthlyCharges': 'mean'
}).round(3)

risk_analysis.columns = ['Count', 'Actual_Churn_Rate', 'Avg_Predicted_Prob', 'Avg_Monthly_Charges']
print(risk_analysis)

# Revenue impact analysis
high_risk_customers = test_df[test_df['risk_level'] == 'High Risk']
revenue_at_risk = high_risk_customers['MonthlyCharges'].sum()
total_revenue = test_df['MonthlyCharges'].sum()

print(f"\n💰 Revenue Impact Analysis:")
print(f"  - High risk customers: {len(high_risk_customers)}")
print(f"  - Monthly revenue at risk: ${revenue_at_risk:,.2f}")
print(f"  - Percentage of total revenue: {(revenue_at_risk/total_revenue)*100:.1f}%")

print(f"\n🎯 Model is ready for production integration!")

🔍 Model Interpretation & Feature Importance
📈 Feature Importance (Gradient Boosting):
----------------------------------------
  Contract            : 0.463
  MonthlyCharges      : 0.215
  tenure              : 0.189
  TotalCharges        : 0.067
  AvgChargesPerMonth  : 0.042
  SeniorCitizen       : 0.015
  Dependents          : 0.004
  gender              : 0.003
  SeniorWithFamily    : 0.001
  Partner             : 0.000

💡 Key Business Insights:

📊 Risk Segmentation Analysis:
             Count  Actual_Churn_Rate  Avg_Predicted_Prob  Avg_Monthly_Charges
risk_level                                                                    
Low Risk       884              0.109               0.103               57.379
Medium Risk    423              0.475               0.478               73.518
High Risk      102              0.755               0.782               83.140

💰 Revenue Impact Analysis:
  - High risk customers: 102
  - Monthly revenue at risk: $8,480.25
  - Percentage of total r

C:\Users\athar\AppData\Local\Temp\ipykernel_24460\3817500451.py:51: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  risk_analysis = test_df.groupby('risk_level').agg({


In [22]:
# Test Individual Predictions & Save Model
import joblib
import os

print("🧪 Testing Individual Predictions")
print("="*40)

# Test on a sample customer
sample_customer = X_test.iloc[0:1]  # Take first test customer
actual_churn = y_test.iloc[0]

print(f"📋 Sample Customer Profile:")
for feature, value in sample_customer.iloc[0].items():
    print(f"  {feature}: {value}")

# Make prediction
if 'Logistic Regression' in best_model_name:
    sample_scaled = scaler.transform(sample_customer)
    churn_prob = best_model.predict_proba(sample_scaled)[0, 1]
    churn_pred = best_model.predict(sample_scaled)[0]
else:
    churn_prob = best_model.predict_proba(sample_customer)[0, 1]
    churn_pred = best_model.predict(sample_customer)[0]

risk_level = "High Risk" if churn_prob > 0.7 else "Medium Risk" if churn_prob > 0.3 else "Low Risk"

print(f"\n🎯 Prediction Results:")
print(f"  - Churn Probability: {churn_prob:.1%}")
print(f"  - Risk Level: {risk_level}")
print(f"  - Predicted Churn: {'Yes' if churn_pred else 'No'}")
print(f"  - Actual Churn: {'Yes' if actual_churn else 'No'}")
print(f"  - Prediction Correct: {'✅' if (churn_pred == actual_churn) else '❌'}")

# Business recommendations based on prediction
print(f"\n💼 Business Recommendations:")
if churn_prob > 0.7:
    recommendations = [
        "🔴 URGENT: Immediate retention campaign required",
        "💰 Offer significant discount or upgrade incentive", 
        "📞 Personal outreach from customer success team"
    ]
elif churn_prob > 0.4:
    recommendations = [
        "⚠️ MEDIUM RISK: Proactive retention needed",
        "📧 Send targeted email campaign",
        "💬 Schedule customer satisfaction survey"
    ]
else:
    recommendations = [
        "✅ LOW RISK: Focus on expansion opportunities",
        "🚀 Introduce premium features or add-ons",
        "🎯 Request referrals or case study participation"
    ]

for rec in recommendations:
    print(f"  {rec}")

# Save the trained model and preprocessing components
print(f"\n💾 Saving Model for Production...")

# Create models directory if it doesn't exist
models_dir = "../../ml_models/churn"
os.makedirs(models_dir, exist_ok=True)

# Save model and preprocessing components
model_artifacts = {
    'model': best_model,
    'scaler': scaler if 'Logistic Regression' in best_model_name else None,
    'feature_columns': feature_columns,
    'model_name': best_model_name,
    'performance_metrics': results[best_model_name]
}

model_path = os.path.join(models_dir, 'churn_model.joblib')
joblib.dump(model_artifacts, model_path)

print(f"✅ Model saved to: {model_path}")
print(f"📋 Artifacts saved:")
print(f"  - Trained {best_model_name} model")
print(f"  - Feature preprocessing (scaler)")
print(f"  - Feature column definitions")
print(f"  - Performance metrics")

print(f"\n🚀 CHURN PREDICTION MODEL COMPLETE!")
print("="*50)
print("✅ Model trained and evaluated")
print("✅ Feature engineering pipeline created") 
print("✅ Business insights generated")
print("✅ Model saved for production")
print("\n🎯 Ready to integrate with FastAPI backend!")

🧪 Testing Individual Predictions
📋 Sample Customer Profile:
  gender: 1.0
  SeniorCitizen: 0.0
  Partner: 1.0
  Dependents: 1.0
  tenure: 72.0
  Contract: 2.0
  MonthlyCharges: 114.05
  TotalCharges: 8468.2
  AvgChargesPerMonth: 116.00273972602741
  TenureGroup: 2.0
  IsHighValue: 1.0
  SeniorWithFamily: 0.0

🎯 Prediction Results:
  - Churn Probability: 2.2%
  - Risk Level: Low Risk
  - Predicted Churn: No
  - Actual Churn: No
  - Prediction Correct: ✅

💼 Business Recommendations:
  ✅ LOW RISK: Focus on expansion opportunities
  🚀 Introduce premium features or add-ons
  🎯 Request referrals or case study participation

💾 Saving Model for Production...
✅ Model saved to: ../../ml_models/churn\churn_model.joblib
📋 Artifacts saved:
  - Trained Gradient Boosting model
  - Feature preprocessing (scaler)
  - Feature column definitions
  - Performance metrics

🚀 CHURN PREDICTION MODEL COMPLETE!
✅ Model trained and evaluated
✅ Feature engineering pipeline created
✅ Business insights generated
✅ 

In [25]:
# ADVANCED MODEL IMPROVEMENTS
print("🚀 ADVANCED MODEL OPTIMIZATION")
print("="*50)

# 1. Hyperparameter Tuning with Grid Search
print("🔧 Hyperparameter Tuning...")

# Grid search for Gradient Boosting (best performing model)
gb_param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1, 0.2], 
    'max_depth': [3, 5, 7],
    'subsample': [0.8, 1.0]
}

gb_grid_search = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    gb_param_grid,
    cv=5,
    scoring='roc_auc',  # Optimize for AUC
    n_jobs=-1,
    verbose=1
)

print("  📊 Training grid search (this may take a few minutes)...")
gb_grid_search.fit(X_train, y_train)

print(f"  🏆 Best parameters: {gb_grid_search.best_params_}")
print(f"  📈 Best CV AUC: {gb_grid_search.best_score_:.3f}")

# Get optimized model
optimized_gb = gb_grid_search.best_estimator_
optimized_pred = optimized_gb.predict(X_test)
optimized_pred_proba = optimized_gb.predict_proba(X_test)[:, 1]

# Compare with original model
print(f"\n📊 Performance Improvement:")
print(f"  Original GB AUC: {results['Gradient Boosting']['auc']:.3f}")
print(f"  Optimized GB AUC: {roc_auc_score(y_test, optimized_pred_proba):.3f}")
print(f"  Improvement: +{roc_auc_score(y_test, optimized_pred_proba) - results['Gradient Boosting']['auc']:.3f}")

# Update best model
best_model = optimized_gb
best_model_name = "Optimized Gradient Boosting"

🚀 ADVANCED MODEL OPTIMIZATION
🔧 Hyperparameter Tuning...
  📊 Training grid search (this may take a few minutes)...
Fitting 5 folds for each of 36 candidates, totalling 180 fits
  🏆 Best parameters: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.8}
  📈 Best CV AUC: 0.838

📊 Performance Improvement:
  Original GB AUC: 0.836
  Optimized GB AUC: 0.840
  Improvement: +0.004


In [27]:
# 2. Advanced Feature Engineering
print(f"\n⚙️ Advanced Feature Engineering...")

# Add to existing features
X_advanced = X.copy()

# 1. Interaction features (combining important features)
X_advanced['Contract_Tenure_Interaction'] = X_advanced['Contract'] * X_advanced['tenure']
X_advanced['Charges_Tenure_Ratio'] = X_advanced['MonthlyCharges'] / (X_advanced['tenure'] + 1)

# 2. Binned features (transform continuous to categorical insights)
X_advanced['TenureGroup_Detailed'] = pd.cut(X_advanced['tenure'], 
                                          bins=[0, 6, 12, 24, 36, 100], 
                                          labels=[0, 1, 2, 3, 4])
X_advanced['TenureGroup_Detailed'] = X_advanced['TenureGroup_Detailed'].astype('category').cat.codes

# 3. Revenue-based features
X_advanced['Revenue_Per_Month_Category'] = pd.cut(X_advanced['MonthlyCharges'],
                                                 bins=3, labels=[0, 1, 2]).astype(int)

# 4. Customer lifetime value proxy  
X_advanced['CustomerValue_Proxy'] = (X_advanced['tenure'] * X_advanced['MonthlyCharges']) / 100

# Update feature columns
advanced_feature_columns = list(X_advanced.columns)
print(f"  🔧 Original features: {len(feature_columns)}")
print(f"  🚀 Advanced features: {len(advanced_feature_columns)}")
print(f"  ➕ New features added: {len(advanced_feature_columns) - len(feature_columns)}")

# Re-split data with advanced features
X_train_adv, X_test_adv, y_train_adv, y_test_adv = train_test_split(
    X_advanced, y, test_size=0.2, random_state=42, stratify=y
)

print(f"  ✅ Advanced feature engineering complete!")


⚙️ Advanced Feature Engineering...
  🔧 Original features: 12
  🚀 Advanced features: 17
  ➕ New features added: 5
  ✅ Advanced feature engineering complete!


In [28]:
# 3. Train Advanced Model with New Features
print(f"\n🎯 Training with Advanced Features...")

# Train optimized model with advanced features
final_model = GradientBoostingClassifier(**gb_grid_search.best_params_, random_state=42)
final_model.fit(X_train_adv, y_train_adv)

# Predictions
final_pred_proba = final_model.predict_proba(X_test_adv)[:, 1]
final_pred = final_model.predict(X_test_adv)

# Performance metrics
final_auc = roc_auc_score(y_test_adv, final_pred_proba)
final_accuracy = accuracy_score(y_test_adv, final_pred)
final_precision = precision_score(y_test_adv, final_pred)
final_recall = recall_score(y_test_adv, final_pred)

print(f"🏆 FINAL MODEL PERFORMANCE:")
print(f"  - AUC: {final_auc:.3f}")
print(f"  - Accuracy: {final_accuracy:.3f}")
print(f"  - Precision: {final_precision:.3f}")
print(f"  - Recall: {final_recall:.3f}")

# Feature importance for new model
final_feature_importance = pd.DataFrame({
    'feature': advanced_feature_columns,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\n📈 TOP 10 MOST IMPORTANT FEATURES:")
for _, row in final_feature_importance.head(10).iterrows():
    print(f"  {row['feature']:<25}: {row['importance']:.3f}")

# Compare all models
print(f"\n📊 MODEL COMPARISON SUMMARY:")
print(f"  Original GB AUC:     {results['Gradient Boosting']['auc']:.3f}")
print(f"  Optimized GB AUC:    {roc_auc_score(y_test, optimized_pred_proba):.3f}")
print(f"  Final Advanced AUC:  {final_auc:.3f}")

improvement = final_auc - results['Gradient Boosting']['auc']
print(f"  🚀 Total Improvement: +{improvement:.3f} ({improvement/results['Gradient Boosting']['auc']*100:.1f}%)")


🎯 Training with Advanced Features...
🏆 FINAL MODEL PERFORMANCE:
  - AUC: 0.841
  - Accuracy: 0.800
  - Precision: 0.669
  - Recall: 0.487

📈 TOP 10 MOST IMPORTANT FEATURES:
  Charges_Tenure_Ratio     : 0.470
  Contract                 : 0.247
  MonthlyCharges           : 0.173
  AvgChargesPerMonth       : 0.029
  Contract_Tenure_Interaction: 0.022
  TotalCharges             : 0.021
  SeniorCitizen            : 0.014
  CustomerValue_Proxy      : 0.010
  Dependents               : 0.005
  tenure                   : 0.004

📊 MODEL COMPARISON SUMMARY:
  Original GB AUC:     0.836
  Optimized GB AUC:    0.840
  Final Advanced AUC:  0.841
  🚀 Total Improvement: +0.005 (0.6%)


In [29]:
# 4. Business-Optimized Threshold Selection
print(f"\n💼 Business-Optimized Threshold Selection")
print("="*50)

# Business cost assumptions (you can adjust these)
RETENTION_COST = 50      # Cost to run retention campaign per customer
CHURN_LOSS = 500        # Average revenue lost if customer churns (annual value)
CAMPAIGN_SUCCESS_RATE = 0.3  # 30% of targeted customers are saved

# Test different thresholds
thresholds = np.arange(0.1, 0.9, 0.05)
business_results = []

for threshold in thresholds:
    # Predict with custom threshold
    preds = (final_pred_proba >= threshold).astype(int)
    
    # Confusion matrix components
    tp = np.sum((preds == 1) & (y_test_adv == 1))  # True positives - correctly identified churners
    fp = np.sum((preds == 1) & (y_test_adv == 0))  # False positives - campaigns to non-churners  
    fn = np.sum((preds == 0) & (y_test_adv == 1))  # False negatives - missed churners
    tn = np.sum((preds == 0) & (y_test_adv == 0))  # True negatives - correctly identified loyals
    
    # Business calculations
    campaign_cost = (tp + fp) * RETENTION_COST  # Cost to target all predicted churners
    revenue_saved = tp * CAMPAIGN_SUCCESS_RATE * CHURN_LOSS  # Revenue saved from successful campaigns
    revenue_lost = fn * CHURN_LOSS  # Revenue lost from missed churners
    
    total_cost = campaign_cost + revenue_lost
    net_benefit = revenue_saved - total_cost
    
    # Metrics
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    
    business_results.append({
        'threshold': threshold,
        'precision': precision,
        'recall': recall,
        'campaign_cost': campaign_cost,
        'revenue_saved': revenue_saved,
        'revenue_lost': revenue_lost,
        'net_benefit': net_benefit,
        'customers_targeted': tp + fp,
        'churners_caught': tp,
        'churners_missed': fn
    })

# Convert to DataFrame for analysis
business_df = pd.DataFrame(business_results)

# Find optimal threshold (maximize net benefit)
optimal_idx = business_df['net_benefit'].idxmax()
optimal_threshold = business_df.loc[optimal_idx, 'threshold']
optimal_net_benefit = business_df.loc[optimal_idx, 'net_benefit']

print(f"🎯 BUSINESS-OPTIMIZED RESULTS:")
print(f"  Optimal Threshold: {optimal_threshold:.2f}")
print(f"  Expected Net Benefit: ${optimal_net_benefit:,.2f}")
print(f"  Customers to Target: {business_df.loc[optimal_idx, 'customers_targeted']:.0f}")
print(f"  Expected Churners Saved: {business_df.loc[optimal_idx, 'churners_caught'] * CAMPAIGN_SUCCESS_RATE:.1f}")
print(f"  Churners Missed: {business_df.loc[optimal_idx, 'churners_missed']:.0f}")

# Show top 3 thresholds by net benefit
print(f"\n📊 TOP 3 BUSINESS THRESHOLDS:")
top_3 = business_df.nlargest(3, 'net_benefit')[['threshold', 'precision', 'recall', 'net_benefit', 'customers_targeted']]
print(top_3)

# Final business recommendation
final_threshold_pred = (final_pred_proba >= optimal_threshold).astype(int)
final_business_precision = precision_score(y_test_adv, final_threshold_pred)
final_business_recall = recall_score(y_test_adv, final_threshold_pred)

print(f"\n🎯 FINAL BUSINESS-OPTIMIZED MODEL:")
print(f"  Threshold: {optimal_threshold:.2f}")
print(f"  Business Precision: {final_business_precision:.3f}")
print(f"  Business Recall: {final_business_recall:.3f}")
print(f"  Expected ROI: ${optimal_net_benefit:,.2f} per month")


💼 Business-Optimized Threshold Selection
🎯 BUSINESS-OPTIMIZED RESULTS:
  Optimal Threshold: 0.10
  Expected Net Benefit: $-1,600.00
  Customers to Target: 894
  Expected Churners Saved: 106.2
  Churners Missed: 20

📊 TOP 3 BUSINESS THRESHOLDS:
   threshold  precision    recall  net_benefit  customers_targeted
0       0.10   0.395973  0.946524      -1600.0                 894
1       0.15   0.435567  0.903743      -6100.0                 776
2       0.20   0.468242  0.847594     -14800.0                 677

🎯 FINAL BUSINESS-OPTIMIZED MODEL:
  Threshold: 0.10
  Business Precision: 0.396
  Business Recall: 0.947
  Expected ROI: $-1,600.00 per month


In [30]:
# 5. Save Enhanced Model & Final Summary
print(f"\n💾 SAVING ENHANCED MODEL FOR PRODUCTION")
print("="*50)

# Create enhanced model artifacts
enhanced_model_artifacts = {
    'model': final_model,
    'feature_columns': advanced_feature_columns,
    'optimal_threshold': optimal_threshold,
    'scaler': None,  # Not needed for tree-based models
    'model_name': 'Enhanced Gradient Boosting',
    'performance_metrics': {
        'auc': final_auc,
        'accuracy': final_accuracy,
        'precision': final_precision,
        'recall': final_recall,
        'business_precision': final_business_precision,
        'business_recall': final_business_recall,
        'optimal_threshold': optimal_threshold,
        'expected_roi_monthly': optimal_net_benefit
    },
    'feature_importance': final_feature_importance.to_dict(),
    'business_parameters': {
        'retention_cost': RETENTION_COST,
        'churn_loss': CHURN_LOSS,
        'campaign_success_rate': CAMPAIGN_SUCCESS_RATE
    }
}

# Save enhanced model
enhanced_model_path = models_dir + '/enhanced_churn_model.joblib'
joblib.dump(enhanced_model_artifacts, enhanced_model_path)

print(f"✅ Enhanced model saved to: {enhanced_model_path}")

# FINAL SUMMARY REPORT
print(f"\n" + "="*80)
print(f"🏆 CHURN PREDICTION MODEL - FINAL PERFORMANCE REPORT")
print(f"="*80)

print(f"\n📈 MODEL EVOLUTION:")
print(f"  1️⃣  Basic Model (Default GB):        AUC = {results['Gradient Boosting']['auc']:.3f}")
print(f"  2️⃣  Hyperparameter Tuned:           AUC = {roc_auc_score(y_test, optimized_pred_proba):.3f}")
print(f"  3️⃣  Advanced Features + Tuned:      AUC = {final_auc:.3f}")
print(f"  🚀 Total Improvement:               +{improvement:.3f} ({improvement/results['Gradient Boosting']['auc']*100:.1f}%)")

print(f"\n🎯 FINAL MODEL SPECIFICATIONS:")
print(f"  • Algorithm: {final_model.__class__.__name__}")
print(f"  • Features: {len(advanced_feature_columns)} (vs {len(feature_columns)} original)")
print(f"  • Training Samples: {len(X_train_adv):,}")
print(f"  • Test Samples: {len(X_test_adv):,}")

print(f"\n📊 TECHNICAL PERFORMANCE:")
print(f"  • AUC Score: {final_auc:.3f}")
print(f"  • Accuracy: {final_accuracy:.3f}")
print(f"  • Precision: {final_precision:.3f}")
print(f"  • Recall: {final_recall:.3f}")

print(f"\n💼 BUSINESS PERFORMANCE:")
print(f"  • Optimal Threshold: {optimal_threshold:.2f}")
print(f"  • Business Precision: {final_business_precision:.3f}")
print(f"  • Business Recall: {final_business_recall:.3f}")
print(f"  • Expected Monthly ROI: ${optimal_net_benefit:,.2f}")
print(f"  • Revenue at Risk Identified: ${business_df.loc[optimal_idx, 'revenue_saved']:,.2f}")

print(f"\n🔝 TOP PREDICTIVE FEATURES:")
for i, (_, row) in enumerate(final_feature_importance.head(5).iterrows(), 1):
    print(f"  {i}. {row['feature']:<25}: {row['importance']:.3f}")

print(f"\n🎯 NEXT STEPS FOR PRODUCTION:")
print(f"  ✅ Model saved and ready for FastAPI integration")
print(f"  🔌 Use optimal threshold ({optimal_threshold:.2f}) for business predictions")
print(f"  📊 Monitor model performance with actual business results")
print(f"  🔄 Retrain monthly with new data")

print(f"\n" + "="*80)
print(f"🚀 ADVANCED CHURN PREDICTION MODEL COMPLETE!")
print(f"Ready for production deployment 🎉")
print(f"="*80)


💾 SAVING ENHANCED MODEL FOR PRODUCTION
✅ Enhanced model saved to: ../../ml_models/churn/enhanced_churn_model.joblib

🏆 CHURN PREDICTION MODEL - FINAL PERFORMANCE REPORT

📈 MODEL EVOLUTION:
  1️⃣  Basic Model (Default GB):        AUC = 0.836
  2️⃣  Hyperparameter Tuned:           AUC = 0.840
  3️⃣  Advanced Features + Tuned:      AUC = 0.841
  🚀 Total Improvement:               +0.005 (0.6%)

🎯 FINAL MODEL SPECIFICATIONS:
  • Algorithm: GradientBoostingClassifier
  • Features: 17 (vs 12 original)
  • Training Samples: 5,634
  • Test Samples: 1,409

📊 TECHNICAL PERFORMANCE:
  • AUC Score: 0.841
  • Accuracy: 0.800
  • Precision: 0.669
  • Recall: 0.487

💼 BUSINESS PERFORMANCE:
  • Optimal Threshold: 0.10
  • Business Precision: 0.396
  • Business Recall: 0.947
  • Expected Monthly ROI: $-1,600.00
  • Revenue at Risk Identified: $53,100.00

🔝 TOP PREDICTIVE FEATURES:
  1. Charges_Tenure_Ratio     : 0.470
  2. Contract                 : 0.247
  3. MonthlyCharges           : 0.173
  4. Avg